# 01 · 심장병 예측 (이진 분류)

**데이터**: UCI Heart Disease (Cleveland) — 환자 나이, 혈압, 콜레스테롤 등으로 심장병 여부 예측.

**이 노트북의 흐름** (의료 AI의 전형적 파이프라인):
1. 데이터 로드
2. 탐색적 분석 (EDA)
3. 전처리 (결측치, 스케일링)
4. 학습/검증 분할
5. 모델 학습 (Logistic Regression)
6. **평가 — 의료에서 특히 중요**
7. 의사결정 해석

> 💡 의료에서는 '틀린 음성(FN, 병인데 정상이라 함)'이 '틀린 양성(FP)'보다 훨씬 위험한 경우가 많다. 그래서 accuracy 하나만 보면 안 된다.

## 1. 데이터 로드

`fetch_openml`로 UCI Heart Disease를 가져온다. 첫 실행 시 인터넷 필요.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml

np.random.seed(42)

# UCI Heart Disease (Cleveland). as_frame=True면 pandas DataFrame으로 받음.
heart = fetch_openml('heart-disease', version=1, as_frame=True, parser='auto')
df = heart.frame
print(f'shape: {df.shape}')
df.head()

## 2. EDA (Exploratory Data Analysis)

모델 만들기 전에 **데이터를 의심하는** 단계.

In [ ]:
print('dtypes:')
print(df.dtypes)
print('\n결측치:')
print(df.isna().sum())
print('\n타겟 분포:')
print(df['target'].value_counts() if 'target' in df.columns else df.iloc[:, -1].value_counts())

In [ ]:
# 타겟 컬럼 이름 맞추기 (openml 버전에 따라 다를 수 있음)
target_col = 'target' if 'target' in df.columns else df.columns[-1]
print(f'타겟 컬럼: {target_col}')

# 숫자형만 뽑아 히스토그램
numeric = df.select_dtypes(include=[np.number])
numeric.hist(figsize=(12, 8), bins=20)
plt.suptitle('Feature distributions', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 타겟과의 상관관계 (숫자 feature만)
corr = numeric.corr()[target_col].drop(target_col).sort_values()
plt.figure(figsize=(6, 5))
corr.plot(kind='barh')
plt.title('Correlation with target')
plt.show()

## 3. 전처리

- 결측치가 있으면 평균(숫자) / 최빈값(범주)으로 채움
- 범주형은 one-hot encoding
- 숫자형은 StandardScaler

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df.drop(columns=[target_col])
y = df[target_col]

# target이 multi-class로 올 수도 있음 → 이진으로 (0 vs >0)
y = (pd.to_numeric(y, errors='coerce') > 0).astype(int)

# 숫자/범주 분리
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
print(f'숫자 feature: {num_cols}')
print(f'범주 feature: {cat_cols}')

# ColumnTransformer로 숫자와 범주를 다르게 처리
from sklearn.preprocessing import OneHotEncoder
preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'train: {X_train.shape}, test: {X_test.shape}')
print(f'train 양성 비율: {y_train.mean():.2%}')

## 4. 모델 학습 — Logistic Regression

가장 단순한 분류 모델부터 시작. **복잡한 모델을 쓰기 전에 단순한 베이스라인**을 두는 것이 의료 ML의 기본.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)
print('학습 완료')

## 5. 평가

의료 분류의 핵심 지표들:
- **Accuracy**: 전체 중 맞춘 비율
- **Precision**: 양성이라 한 것 중 진짜 양성 (거짓 경보 적을수록 높음)
- **Recall (Sensitivity)**: 실제 양성 중 잡아낸 비율 (**환자를 놓치지 않는 능력**)
- **AUROC**: 임계값에 무관하게 양/음을 얼마나 잘 구분하는가

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve,
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision: {precision_score(y_test, y_pred):.3f}')
print(f'Recall   : {recall_score(y_test, y_pred):.3f}')
print(f'F1       : {f1_score(y_test, y_pred):.3f}')
print(f'AUROC    : {roc_auc_score(y_test, y_proba):.3f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

In [ ]:
# Confusion matrix 시각화
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred No', 'Pred Yes'],
            yticklabels=['True No', 'True Yes'], ax=axes[0])
axes[0].set_title('Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, y_proba):.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. 인사이트

**자문해볼 것**:
- Recall이 Precision보다 낮다면: 모델이 "병이 아니다"쪽으로 치우침 → threshold 낮추면 놓침 줄어듦.
- AUROC 0.85+: 심장병 예측에서 의미 있는 수준.
- **임계값 이동**: `y_proba > 0.3` 처럼 기준을 낮추면 recall↑, precision↓ (병원 오용 비용을 바꿈)

### AI agent에게 물어볼 것
1. "ROC curve에서 왼쪽 위로 가까울수록 좋다고 했는데, 그 점이 수학적으로 뭘 의미해?"
2. "Logistic Regression이 내부적으로 어떤 함수를 학습하는지 수식으로 보여줘"
3. "이 데이터에서 clinical하게 가장 중요한 feature는 뭘까? 모델 계수로 설명해줘"

### 다음 노트북
→ `02_diabetes_regression.ipynb`: 같은 전처리 틀을 **회귀** 문제에 적용